In [1]:
# colocate mode velocity reward training debug notebook

# 1. load model & dataset
# 2. initialize PerTokenAdvantageTrainer (vllm colocate mode )
# 3. train with log. 
# Issues: (1) no reward logging for correct rollout 

In [ ]:
# ── 1. Setup ────────────────────────────────────────────────────────────────
# Run from repo root. Keeps everything in a scratch dir so we can rm -rf safely.
import os, sys, json, shutil, random
from pathlib import Path

REPO = Path(".")
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

OUT = REPO / "output" / "debug_vrl"
if OUT.exists():
    shutil.rmtree(OUT)
OUT.mkdir(parents=True, exist_ok=True)
print("scratch dir:", OUT)

# Toggle: vLLM colocate is what the bug was observed under, but the velocity-
# log code path is identical with use_vllm=False (HF generate). Flip to True
# on a CUDA box to reproduce in the exact training config.
USE_VLLM   = False
MODEL_NAME = "Qwen/Qwen3-0.6B"
N_STEPS    = 3
N_GEN      = 4              # rollouts per prompt
PDB        = 2              # per_device_train_batch_size
GAS        = 2              # gradient_accumulation_steps
MAX_COMP   = 1024
SEED       = 0
random.seed(SEED)

import numpy as np, torch
np.random.seed(SEED); torch.manual_seed(SEED)
print("torch:", torch.__version__, "cuda:", torch.cuda.is_available())


scratch dir: output/debug_vrl
torch: 2.9.1 cuda: False


In [3]:
# ── 2. Dataset (tiny) + model/tokenizer ─────────────────────────────────────
from transformers import AutoTokenizer, AutoModelForCausalLM
from src.game24utils import (
    build_puzzle_pool, bucket_by_difficulty, make_splits, build_datasets,
    correctness_reward, format_reward,
)

puzzles = build_puzzle_pool(max_n=9)
easy, medium, hard = bucket_by_difficulty(puzzles, easy_min=8, hard_max=2)
train_p, eval_p, hard_probe = make_splits(easy, medium, hard, eval_frac=0.10, probe_frac=0.40)
train_ds, eval_ds, _ = build_datasets(train_p, eval_p, hard_probe)

# Shrink to keep the debug fast — we only need a few prompts.
train_ds = train_ds.select(range(min(16, len(train_ds))))
print("train_ds rows:", len(train_ds))
print("sample row:", train_ds[0])

tok   = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.bfloat16)
print("loaded:", MODEL_NAME)


train_ds rows: 16
sample row: {'prompt': [{'content': "You play the Game of 24. Given four numbers, you must write a single arithmetic expression using each number exactly once with + - * / and parentheses that evaluates to 24.\n\nThink step by step. First reason about how to combine the numbers, then on the final line output only the expression after '#### '.\nExample final line: '#### (3+5)*(7-4)'.", 'role': 'system'}, {'content': "Given numbers: 3,3,5,9. Make 24. Think step by step, then give your final expression on the last line after '#### '.", 'role': 'user'}], 'numbers': [3, 3, 5, 9], 'solutions': ['((3+5)/3)*9', '(3+5)/(3/9)', '((3+5)*9)/3', '(3+5)*(9/3)', '3*(5+(9/3))', '(9+3)*(5-3)', '(9*(3+5))/3', '9*((3+5)/3)', '((9/3)+5)*3', '(9/3)*(5+3)', '9/(3/(5+3))', '3*((9/3)+5)', '((5+3)*9)/3', '(5+3)*(9/3)', '(5-3)*(9+3)', '(5+(9/3))*3', '(3+3)*(9-5)', '(9/3)*(3+5)', '9/(3/(3+5))', '(9-5)*(3+3)', '(9*(5+3))/3', '9*((5+3)/3)', '((5+3)/3)*9', '(5+3)/(3/9)', '(5-3)*(3+9)', '(3+9)*(5-3

`torch_dtype` is deprecated! Use `dtype` instead!


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

loaded: Qwen/Qwen3-0.6B


In [4]:
# ── 3. Seed the answer buffer with dataset-provided solutions ───────────────
# Mirrors ablate_game24.py:319-325. Without seeding, buffer.sample returns
# empty for every query → the else-branch ref is empty → ALL rollouts get
# silently skipped at the gate. That would explain an empty velocity_log
# but NOT the observed pattern (0 correct out of 3200 logged), so we want a
# populated buffer to isolate the *correct-rollout* path.
from src.online_buffer import OnlineBuffer
from src.velocity import VelocityRewardComputer

answer_buffer = OnlineBuffer(capacity_per_query=64)
n_seed = 0
for row in train_ds:
    qk = tuple(row["numbers"])
    for expr in (row.get("solutions") or []):
        n_seed += int(answer_buffer.add(qk, expr))
print(f"buffer seeded: {n_seed} entries / {answer_buffer.num_queries()} queries")

vel_computer = VelocityRewardComputer(
    answer_buffer,
    chunk_strategy="uniform",
    chunk_size=16,
    normalize_by_chunk=True,
)


buffer seeded: 412 entries / 16 queries


In [5]:
# ── 3. Seed the answer buffer with dataset-provided solutions ───────────────
# Mirrors ablate_game24.py:319-325. Without seeding, buffer.sample returns
# empty for every query → the else-branch ref is empty → ALL rollouts get
# silently skipped at the gate. That would explain an empty velocity_log
# but NOT the observed pattern (0 correct out of 3200 logged), so we want a
# populated buffer to isolate the *correct-rollout* path.
from src.online_buffer import OnlineBuffer
from src.velocity import VelocityRewardComputer

answer_buffer = OnlineBuffer(capacity_per_query=64)
n_seed = 0
for row in train_ds:
    qk = tuple(row["numbers"])
    for expr in (row.get("solutions") or []):
        n_seed += int(answer_buffer.add(qk, expr))
print(f"buffer seeded: {n_seed} entries / {answer_buffer.num_queries()} queries")

vel_computer = VelocityRewardComputer(
    answer_buffer,
    chunk_strategy="uniform",
    chunk_size=16,
    normalize_by_chunk=True,
)


buffer seeded: 412 entries / 16 queries


In [6]:
# ── 4. Correctness + answer-extraction helpers ──────────────────────────────
# Both `is_correct` and `extract_expr` are now hardened against reward-hacking
# (anchor on `</think>`). They're used by the trainer:
#   • `is_correct`  → decides which rollouts use their own answer as velocity
#                     reference vs. a buffer sample, and gates buffer updates.
#   • `extract_expr` → surfaces the chosen answer expression into the unified
#                     rollouts log via `game24_task_extras`.
from script.ablate_game24 import (
    game24_is_correct,
    game24_query_key,
    game24_task_extras,
)


objc[33776]: Class AVFFrameReceiver is implemented in both /Users/fangyuanyu/anaconda3/lib/python3.11/site-packages/av/.dylibs/libavdevice.61.1.100.dylib (0x2f3d10798) and /Users/fangyuanyu/anaconda3/lib/python3.11/site-packages/cv2/.dylibs/libavdevice.61.3.100.dylib (0x3086743a8). One of the two will be used. Which one is undefined.
objc[33776]: Class AVFAudioReceiver is implemented in both /Users/fangyuanyu/anaconda3/lib/python3.11/site-packages/av/.dylibs/libavdevice.61.1.100.dylib (0x2f3d107e8) and /Users/fangyuanyu/anaconda3/lib/python3.11/site-packages/cv2/.dylibs/libavdevice.61.3.100.dylib (0x3086743f8). One of the two will be used. Which one is undefined.


In [ ]:
# ── 5. Build the trainer ────────────────────────────────────────────────────
# Note on the "ld: cannot find -laio" stderr noise some servers emit here:
# accelerate's `get_accelerator()` triggers deepspeed's AsyncIOBuilder, which
# shells out a tiny test compile linking `-laio`. When `libaio-dev` is missing
# the linker prints to stderr; the check then returns False and deepspeed
# silently runs without async_io. The warning is cosmetic — training is
# unaffected. Permanent fix: `apt-get install -y libaio-dev` (or
# `yum install libaio-devel`). Below we suppress the noise locally by
# redirecting low-level FD 2 across the trainer build only.
import contextlib, os as _os

@contextlib.contextmanager
def _silence_stderr():
    devnull_fd = _os.open(_os.devnull, _os.O_WRONLY)
    saved_fd   = _os.dup(2)
    try:
        _os.dup2(devnull_fd, 2)
        yield
    finally:
        _os.dup2(saved_fd, 2)
        _os.close(devnull_fd); _os.close(saved_fd)

from trl import GRPOConfig
from src.pertoken_trainer import PerTokenAdvantageTrainer

cfg_kw = dict(
    output_dir=str(OUT),
    num_generations=N_GEN,
    max_completion_length=MAX_COMP,
    per_device_train_batch_size=PDB,
    gradient_accumulation_steps=GAS,
    learning_rate=5e-6,
    max_steps=N_STEPS,
    logging_steps=1,
    bf16=True,
    save_strategy="no",
    report_to="none",
    use_vllm=USE_VLLM,
)
if USE_VLLM:
    # vLLM's max_model_length is left UNSET so the server uses the model's
    # full positional cap (Qwen3 = 40k). Setting a low value here would clip
    # long prompts+completions silently.
    cfg_kw.update(vllm_mode="colocate", vllm_gpu_memory_utilization=0.4)

# All per-rollout logging is now done by the trainer itself into exactly two
# files: `rollouts.jsonl` (train) and `eval_rollouts.jsonl` (eval). Reward
# functions only return scalars now — no logging shim in `reward_funcs`.
with _silence_stderr():
    cfg = GRPOConfig(**cfg_kw)
    trainer = PerTokenAdvantageTrainer(
        model=model,
        reward_funcs=[correctness_reward, format_reward],
        args=cfg,
        train_dataset=train_ds,
        processing_class=tok,
        adv_mode="token", adv_n_chunks=8, adv_stride=5,
        velocity_computer=vel_computer,
        is_correct=game24_is_correct,
        query_key_fn=game24_query_key,
        task_extras_fn=game24_task_extras,
    )

# Truncate the two sinks at the start of this debug session.
(OUT / "rollouts.jsonl").write_text("")
(OUT / "eval_rollouts.jsonl").write_text("")
print("trainer ready; logs →")
print(" ", OUT / "rollouts.jsonl")
print(" ", OUT / "eval_rollouts.jsonl")


[2026-05-26 12:39:02,840] [INFO] [real_accelerator.py:222:get_accelerator] Setting ds_accelerator to mps (auto detect)


W0526 12:39:02.985000 33776 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


trainer ready; logs →
  output/debug_vrl/rollouts.jsonl
  output/debug_vrl/eval_rollouts.jsonl


In [8]:
# ── 6. Train a few steps ────────────────────────────────────────────────────
trainer.train()
print("done. files in", OUT, ":")
for p in sorted(OUT.iterdir()):
    print(f"  {p.name:30s}  {p.stat().st_size:>8d} bytes")


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
/Users/fangyuanyu/anaconda3/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
Could not estimate the number of tokens of the input, floating-point operations will not be computed


Step,Training Loss
1,0.000000
2,0.000000
3,0.000000


done. files in output/debug_vrl :
  eval_rollouts.jsonl                    0 bytes
  rollouts.jsonl                    203676 bytes


In [9]:
# ── 7. Read back the unified train log ─────────────────────────────────────
def load_jsonl(p):
    return [json.loads(l) for l in Path(p).read_text().splitlines() if l.strip()]

ro      = load_jsonl(OUT / "rollouts.jsonl")
ro_eval = load_jsonl(OUT / "eval_rollouts.jsonl") if (OUT / "eval_rollouts.jsonl").exists() else []

print(f"rollouts.jsonl       : {len(ro):4d}  correct={sum(r['correct'] for r in ro)}")
print(f"eval_rollouts.jsonl  : {len(ro_eval):4d}  correct={sum(r['correct'] for r in ro_eval)}")
if ro:
    n_with_rt  = sum(bool(r.get('r_t'))      for r in ro)
    n_with_adv = sum(bool(r.get('advantage')) for r in ro)
    print(f"  train rollouts with r_t      : {n_with_rt}/{len(ro)}")
    print(f"  train rollouts with advantage: {n_with_adv}/{len(ro)}")
    sample = ro[0]
    print(f"\nschema (first record keys): {list(sample.keys())}")
    print(f"  global_step={sample['global_step']} inner_step={sample['inner_step']} "
          f"idx={sample['idx']} split={sample['split']}")
    print(f"  numbers={sample.get('numbers')} expr={sample.get('expr')!r} correct={sample['correct']}")
    print(f"  n_tokens={sample['n_tokens']}  len(r_t)={len(sample['r_t'])}  "
          f"len(advantage)={len(sample['advantage'])}")


rollouts.jsonl       :   12  correct=0
eval_rollouts.jsonl  :    0  correct=0
  train rollouts with r_t      : 12/12
  train rollouts with advantage: 12/12

schema (first record keys): ['global_step', 'inner_step', 'idx', 'split', 'completion', 'n_tokens', 'r_t', 'advantage', 'loss', 'correct', 'numbers', 'expr']
  global_step=0 inner_step=0 idx=0 split=train
  numbers=[2, 4, 4, 8] expr='' correct=False
  n_tokens=256  len(r_t)=256  len(advantage)=256


In [10]:
# ── 8. Diagnose: why is the loss ~0?  Per-step advantage variance check ────
# Advantages z-score within a single `_compute_loss` call → all records sharing
# `(global_step, inner_step)` form one pool whose advantages sum to ~0 over
# real tokens. If that pool's std is also 0 (everyone got the same reward),
# every advantage is 0 → loss is 0 → no gradient. This cell checks for that.
import numpy as _np
from collections import defaultdict

pools = defaultdict(list)  # (gs, inner) -> list of (R_T, n_tokens, correct)
for r in ro:
    key = (r["global_step"], r["inner_step"])
    rt  = r.get("r_t") or []
    pools[key].append((float(sum(rt)), int(r["n_tokens"]), bool(r["correct"])))

print(f"{'(gs, inner)':>14s}  {'n':>3s}  {'#correct':>8s}  {'R_T mean':>10s}  {'R_T std':>10s}  {'len mean':>9s}")
flat = 0
for k in sorted(pools):
    vs   = pools[k]
    rts  = _np.array([v[0] for v in vs])
    lens = _np.array([v[1] for v in vs])
    n_ok = sum(v[2] for v in vs)
    sd   = float(rts.std(ddof=0))
    flat += int(sd == 0)
    print(f"  ({k[0]:>3d},{k[1]:>3d})  {len(vs):>3d}  {n_ok:>8d}  {rts.mean():>+10.3f}  {sd:>10.3f}  {lens.mean():>9.1f}")

print(f"\npools with std==0 (zero-gradient cause): {flat}/{len(pools)}")
if flat == len(pools):
    print("→ every microbatch is flat. Loss ≈ 0 is expected (no learning signal).")


   (gs, inner)    n  #correct    R_T mean     R_T std   len mean
  (  0,  0)    2         0      -1.532       3.779      256.0
  (  0,  1)    2         0      +0.794       4.730      256.0
  (  1,  0)    2         0     -14.953       5.437      256.0
  (  1,  1)    2         0      -5.273       1.075      256.0
  (  2,  0)    2         0      -2.897       2.124      256.0
  (  2,  1)    2         0      +3.411       6.039      256.0

pools with std==0 (zero-gradient cause): 0/6


In [11]:
# ── 9. extract_expr coverage on correct rollouts (sanity) ──────────────────
# For rollouts the trainer flagged correct, confirm the canonical extractor
# can recover an answer. A miss here means correctness used a different
# path (e.g. dataset solutions) than the velocity computer's extract_answer
# — that would be the smoking gun for the original "0 correct in log" bug.
from src.game24utils import extract_expr

correct_rollouts = [r for r in ro if r["correct"]]
print(f"train rollouts marked correct: {len(correct_rollouts)}")

n_match = 0; sample_miss = None
for r in correct_rollouts:
    ids = tok.encode(r["completion"], add_special_tokens=False)
    comp_str = tok.decode(ids, skip_special_tokens=False)
    if extract_expr(comp_str):
        n_match += 1
    elif sample_miss is None:
        sample_miss = comp_str

print(f"extract_expr matches on {n_match}/{len(correct_rollouts)}")
if sample_miss is not None:
    print("\nFirst miss (re-encoded completion tail):")
    print(repr(sample_miss[-300:]))


train rollouts marked correct: 0
extract_expr matches on 0/0
